In [5]:
# ==============================
# Sports Injury Risk Prediction
# SVM + RFE Replication
# ==============================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------
# Load Dataset
# ------------------------------
df = pd.read_csv("/content/collegiate_athlete_injury_dataset.csv")

# ------------------------------
# Define Target Variable
# ------------------------------
y = df["Injury_Indicator"]

# ------------------------------
# Select Numeric Features Only
# ------------------------------
X = df.drop(
    columns=["Injury_Indicator", "Athlete_ID", "Gender", "Position"]
)

# ------------------------------
# Standardization
# ------------------------------
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# ------------------------------
# Train / Validation / Test Split
# 70% / 15% / 15%
# ------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# ------------------------------
# Recursive Feature Elimination
# ------------------------------
svm_linear = SVC(kernel="linear")
rfe = RFE(svm_linear, n_features_to_select=10)

X_train_rfe = rfe.fit_transform(X_train, y_train)
X_val_rfe = rfe.transform(X_val)
X_test_rfe = rfe.transform(X_test)

# ------------------------------
# SVM with RBF Kernel
# ------------------------------
svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True)
svm_model.fit(X_train_rfe, y_train)

# ------------------------------
# Evaluation
# ------------------------------
y_pred = svm_model.predict(X_test_rfe)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[28  0]
 [ 1  1]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        28
           1       1.00      0.50      0.67         2

    accuracy                           0.97        30
   macro avg       0.98      0.75      0.82        30
weighted avg       0.97      0.97      0.96        30

